# DEVKIT AI — DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic
## Google Colab + Ollama + Cloudflare Tunnel

Notebook này triển khai model Hugging Face bằng **Ollama + GGUF Q4_K_M**, sau đó public Ollama API tạm thời qua Cloudflare Quick Tunnel để kết nối VS Code/Cline/OpenCode.

**Model gốc:** `xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic`  
**GGUF dùng để chạy:** `mradermacher/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-i1-GGUF:Q4_K_M`  
**Kích thước quant Q4_K_M:** khoảng 5.63 GB.

> Chọn Runtime → Change runtime type → T4 GPU (hoặc GPU khác), bật Internet. Runtime miễn phí có giới hạn thời gian và quota. Không cần tải 18.8 GB BF16 gốc để chạy notebook này.

**Lưu ý bảo mật:** Cloudflare Quick Tunnel tạo URL công khai. Ollama không tự yêu cầu API key cho endpoint này. Chỉ dùng thử nghiệm, không đưa dữ liệu riêng tư/production lên tunnel và dừng tunnel sau khi dùng.

## 1. Kiểm tra GPU và dung lượng

Nếu `nvidia-smi` không thấy GPU, hãy bật GPU trong Runtime settings rồi chạy lại cell.

In [ ]:
!nvidia-smi
!df -h /

## 2. Cài đặt tiện ích và Ollama

In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y zstd curl
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

## 3. Khởi động Ollama server

Giữ Ollama lắng nghe trên localhost trong runtime Colab. Không expose trực tiếp port 11434 ra Internet.

In [ ]:
import os, subprocess, time, requests, socket

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
ollama_log = open("/content/ollama.log", "w")
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    env=os.environ.copy(),
    stdout=ollama_log,
    stderr=subprocess.STDOUT
)

for _ in range(30):
    try:
        r = requests.get("http://127.0.0.1:11434", timeout=2)
        if r.status_code == 200:
            print("Ollama server is ready:", r.text)
            break
    except requests.RequestException:
        time.sleep(2)
else:
    raise RuntimeError("Ollama không khởi động được. Xem /content/ollama.log")

## 4. Tải model GGUF Q4_K_M

Bản Hugging Face gốc là Safetensors BF16 khoảng 18.8 GB. Để chạy thuận tiện hơn trên GPU Colab, notebook dùng bản GGUF quantized Q4_K_M (~5.63 GB), được tác giả quantization cung cấp và có hướng dẫn Ollama trực tiếp.

Có thể mất vài phút tùy tốc độ mạng.

In [ ]:
MODEL = "hf.co/mradermacher/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-i1-GGUF:Q4_K_M"
!ollama pull hf.co/mradermacher/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-i1-GGUF:Q4_K_M
!ollama list

## 5. Kiểm tra model bằng Ollama API

In [ ]:
import requests, json

MODEL = "hf.co/mradermacher/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-i1-GGUF:Q4_K_M"
payload = {
    "model": MODEL,
    "messages": [
        {"role": "user", "content": "Reply in Vietnamese. Giới thiệu ngắn gọn khả năng của bạn."}
    ],
    "stream": False,
    "options": {"num_ctx": 8192, "temperature": 0.7}
}
resp = requests.post("http://127.0.0.1:11434/api/chat", json=payload, timeout=600)
print("HTTP:", resp.status_code)
resp.raise_for_status()
print(resp.json()["message"]["content"])

## 6. Cài đặt Cloudflared

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /content/cloudflared.deb
!sudo dpkg -i /content/cloudflared.deb
!cloudflared --version

## 7. Tạo Cloudflare Quick Tunnel

Cell này tạo URL `https://....trycloudflare.com`. Hãy giữ cell chạy trong suốt thời gian sử dụng. URL sẽ thay đổi khi tạo tunnel mới.

In [ ]:
import re, subprocess, time, os

log_path = "/content/cloudflared.log"
cloudflared_log = open(log_path, "w")
cloudflared_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434"],
    stdout=cloudflared_log,
    stderr=subprocess.STDOUT
)

PUBLIC_URL = None
for _ in range(60):
    time.sleep(2)
    cloudflared_log.flush()
    try:
        log_text = open(log_path, "r").read()
    except Exception:
        log_text = ""
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        PUBLIC_URL = match.group(0)
        break

if PUBLIC_URL:
    print("PUBLIC OLLAMA URL:", PUBLIC_URL)
    print("OpenAI-compatible Base URL:", PUBLIC_URL + "/v1")
else:
    print(open(log_path, "r").read()[-5000:])
    raise RuntimeError("Không lấy được URL Quick Tunnel. Kiểm tra log hoặc thử chạy lại.")

## 8. Kiểm tra public endpoint

Endpoint này công khai trong thời gian tunnel hoạt động. Không chia sẻ URL nếu không cần thiết.

In [ ]:
import requests

test = requests.get(PUBLIC_URL + "/api/tags", timeout=60)
print("HTTP:", test.status_code)
print(test.text[:4000])
test.raise_for_status()

## 9. Kiểm tra OpenAI-compatible API qua tunnel

Dùng `/v1/chat/completions` để kiểm tra khả năng tương thích với các công cụ hỗ trợ OpenAI API.

In [ ]:
openai_payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Reply exactly: DEVKIT AI tunnel test successful."}],
    "stream": False
}
r = requests.post(
    PUBLIC_URL + "/v1/chat/completions",
    json=openai_payload,
    headers={"Authorization": "Bearer ollama"},
    timeout=600
)
print("HTTP:", r.status_code)
print(r.text[:4000])
r.raise_for_status()

## 10. Cấu hình VS Code / Cline / OpenAI-compatible client

Điền thông tin sau vào công cụ client:

| Trường | Giá trị |
|---|---|
| Provider | OpenAI Compatible / OpenAI Compatible API |
| Base URL | `https://<quick-tunnel>.trycloudflare.com/v1` |
| API Key | `ollama` (giá trị placeholder; Ollama local không xác thực key) |
| Model ID | `hf.co/mradermacher/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-i1-GGUF:Q4_K_M` |

Thay `<quick-tunnel>` bằng hostname vừa in ở cell 7. Nếu client yêu cầu model ID, dùng đúng tên model hiển thị trong `ollama list`.

**Quan trọng:** Một số AI Coding Agent cần tool calling/streaming hoặc endpoint riêng. Notebook chỉ xác nhận API chat cơ bản; cần kiểm tra thực tế từng client.

## 11. Xử lý sự cố

- **Không có GPU:** Runtime → Change runtime type → GPU, sau đó chạy lại `!nvidia-smi`.
- **Hết VRAM:** giảm context xuống 4096/8192; đóng các tiến trình GPU khác; thử quant thấp hơn nếu có.
- **Không tải được model:** kiểm tra Internet, quota đĩa và chạy lại `ollama pull`.
- **Tunnel không có URL:** xem `/content/cloudflared.log`, khởi động lại cell 7.
- **Client không kết nối:** kiểm tra URL còn hoạt động, Base URL có `/v1`, model ID chính xác.
- **Runtime bị reset:** Ollama, model cache và tunnel có thể cần khởi động/tải lại tùy trạng thái lưu trữ của Colab.

## 12. Dừng dịch vụ và lưu ý bảo mật

Chạy cell này khi kết thúc. Sau khi dừng tunnel, URL public không còn phục vụ phiên Ollama này.

In [ ]:
for p in [cloudflared_process, ollama_process]:
    try:
        p.terminate()
        p.wait(timeout=10)
    except Exception:
        try:
            p.kill()
        except Exception:
            pass

print("Đã gửi lệnh dừng Ollama và Cloudflare Tunnel.")

---
**DEVKIT AI** — Google Colab + Ollama + Cloudflare Tunnel

Nguồn model:
- Model card: https://huggingface.co/xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic
- GGUF/Ollama quant: https://huggingface.co/mradermacher/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-i1-GGUF

Model card mô tả đây là bản Heretic/decensored của model distill Qwen3.5-9B từ DeepSeek-V4-Pro. Các benchmark và tuyên bố hiệu năng trong model card là do nhà phát hành cung cấp, không phải kết quả kiểm thử của notebook này.